# Linear Regression: Exercise Solutions

---

This notebook contains worked solutions to the nine exercises of the *Linear Regression* notebook (`01_EAIF_Linear_Models_1.ipynb`). Each solution includes:
- Complete working code
- An explanation of the approach
- A business reading of the results, written from the numbers the code actually produces

**Prerequisites:** none. The notebook is self-contained: the first two code cells reload the banking data and refit the banking model exactly as the main notebook does, and Exercise 6 reloads the Bitcoin data. Numbers quoted in the text are from one run of this notebook ("in this run"); a rerun on Colab can differ in the last digit.


In [ ]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn import metrics
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Recreate the banking dataset and model from the main notebook
# This ensures we have all necessary variables for the exercises

banking_url = "https://raw.githubusercontent.com/umatter/EDFB/main/data/banking.csv"
dataset = pd.read_csv(banking_url)

# Prepare the data exactly as in the main notebook
# Leakage rule: 'duration' is the target, 'y' and 'campaign' are only known during or after the call, so they are excluded.
df_banking = dataset.copy()
df_banking['was_previously_contacted'] = (df_banking['pdays'] != 999).astype(int)
df_banking['pdays_clean'] = df_banking['pdays'].replace(999, np.nan)
df_banking['pdays_clean'] = df_banking['pdays_clean'].fillna(df_banking['pdays_clean'].median())   # median of the full data, as in the main notebook (simplification)

feature_cols_cat = ['marital', 'education', 'housing', 'loan', 'contact', 'poutcome']
feature_cols_num = ['age', 'previous', 'pdays_clean', 'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed', 'was_previously_contacted']
X_df_banking = pd.get_dummies(df_banking[feature_cols_cat + feature_cols_num], drop_first=True).astype(float)
# 'housing' and 'loan' are 'unknown' for the same 990 clients: identical dummy columns, so keep only one of them
X_df_banking = X_df_banking.drop(columns=['loan_unknown'])
X_banking = X_df_banking.values
y_banking = np.log1p(df_banking['duration']).values.reshape(-1, 1)

# Split the data
X_train_banking, X_test_banking, y_train_banking, y_test_banking = train_test_split(
    X_banking, y_banking, test_size=0.2, random_state=0
)

# Train the model
model_banking = LinearRegression()
model_banking.fit(X_train_banking, y_train_banking)
y_test_predicted_banking = model_banking.predict(X_test_banking)

print("Banking dataset prepared successfully!")
print(f"Features: {X_df_banking.shape[1]}")
print(f"Training samples: {X_train_banking.shape[0]}")
print(f"Test samples: {X_test_banking.shape[0]}")
print(f"Test R-squared: {r2_score(y_test_banking, y_test_predicted_banking):.4f}")


## Exercise 1: Which coefficients are large and precisely estimated?

**Task:** Interpret the coefficients of the banking model. A coefficient deserves a business sentence only if it is both large enough to matter *and* estimated precisely enough to trust.

### Solution Approach:
1. Refit the same model with `statsmodels`, which reports standard errors and confidence intervals (scikit-learn's `LinearRegression` gives the identical coefficients but no uncertainty)
2. Put coefficient, standard error and 95% confidence interval in one labelled table, sorted by absolute size
3. Keep the coefficients that are large (|coef| > 0.05, i.e. more than about a 5% effect on duration) *and* whose confidence interval excludes 0
4. Read the survivors in business terms, remembering the VIF result of the main notebook


In [ ]:
# Exercise 1 Solution

# Refit with statsmodels to get standard errors and confidence intervals
X_train_sm1 = sm.add_constant(X_train_banking)
model_sm1 = sm.OLS(y_train_banking.ravel(), X_train_sm1).fit()

coef_df = pd.DataFrame({
    'Feature': ['const'] + list(X_df_banking.columns),
    'Coefficient': model_sm1.params,
    'Std_Error': model_sm1.bse,
    'CI_low': model_sm1.conf_int()[:, 0],
    'CI_high': model_sm1.conf_int()[:, 1],
    'p_value': model_sm1.pvalues,
})
coef_df = coef_df[coef_df['Feature'] != 'const'].copy()
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
coef_df['precise'] = (coef_df['CI_low'] > 0) | (coef_df['CI_high'] < 0)   # confidence interval excludes 0
coef_df['large'] = coef_df['Abs_Coefficient'] > 0.05

coef_df_sorted = coef_df.sort_values('Abs_Coefficient', ascending=False)

print("=== ALL COEFFICIENTS (sorted by absolute size) ===")
print(f"Intercept: {model_sm1.params[0]:.4f}")
print(coef_df_sorted[['Feature', 'Coefficient', 'Std_Error', 'CI_low', 'CI_high', 'p_value', 'large', 'precise']]
      .to_string(index=False, float_format='%.4f'))

print("\n=== LARGE AND PRECISELY ESTIMATED (|coef| > 0.05 and CI excludes 0) ===")
keep = coef_df_sorted[coef_df_sorted['large'] & coef_df_sorted['precise']]
print(keep[['Feature', 'Coefficient', 'CI_low', 'CI_high']].to_string(index=False, float_format='%.4f'))

print("\n=== LARGE BUT NOT PRECISELY ESTIMATED (CI contains 0) ===")
print(coef_df_sorted[coef_df_sorted['large'] & ~coef_df_sorted['precise']][['Feature', 'Coefficient', 'CI_low', 'CI_high']]
      .to_string(index=False, float_format='%.4f'))


In [ ]:
# Visualize the 15 largest coefficients with their 95% confidence intervals
top_features = coef_df_sorted.head(15).iloc[::-1]   # reverse so the largest is on top

plt.figure(figsize=(10, 8))
colors = ['tab:blue' if p else 'lightgray' for p in top_features['precise']]
plt.barh(top_features['Feature'], top_features['Coefficient'], color=colors, alpha=0.8)
plt.errorbar(top_features['Coefficient'], top_features['Feature'],
             xerr=[top_features['Coefficient'] - top_features['CI_low'], top_features['CI_high'] - top_features['Coefficient']],
             fmt='none', ecolor='black', capsize=3)
plt.axvline(x=0, color='black', linestyle='--', alpha=0.5)
plt.xlabel('Coefficient (effect on log1p(duration)) with 95% confidence interval')
plt.title('15 largest coefficients: blue = confidence interval excludes 0, gray = it does not')
plt.tight_layout()
plt.show()


### Business Interpretation (from this run):

**Large and precise.** Six coefficients pass both tests in this run:

- `contact_telephone` (about -0.21, CI -0.24 to -0.19): calls to a landline are roughly 20% shorter than calls to a mobile, other things equal. The most reliable single finding in the model.
- `euribor3m` (+0.18), `cons_price_idx` (+0.15) and `emp_var_rate` (-0.13): the macro indicators. Each is precisely estimated, but the main notebook's VIF check showed they are strongly collinear (VIFs of 30 to 65), so their *individual* signs and sizes are not separately meaningful; the joint message is that call length moves with the state of the economy, which for staffing purposes means "recalibrate the average call length when the macro environment changes".
- `education_university.degree` (about -0.08) and `education_professional.course` (about -0.08): clients with higher education have calls that are roughly 8% shorter than the reference group (basic 4-year education), presumably because they decide faster.

**Large but imprecise.** `was_previously_contacted` (+0.17), `marital_unknown` (+0.13), `education_illiterate` (+0.06) and `poutcome_success` (+0.07) look large but their confidence intervals contain 0: the groups are small (18 illiterate clients, 80 with unknown marital status) or the variable is collinear with another (`was_previously_contacted` with `poutcome_success`, VIF around 13). No business decision should rest on them.

**What the coefficients cannot do.** Even the reliable ones shift the *average* log duration by 0.1 to 0.2, while the residual standard deviation is about 0.9. They describe which groups are systematically a little longer or shorter on the phone; they do not make individual calls predictable (test R² of about 0.013 in this run).


## Exercise 2: Residual Analysis

**Task:** Perform a comprehensive residual analysis to check model assumptions.

### Solution Approach:
1. Calculate residuals for training and test sets
2. Create diagnostic plots
3. Test for normality
4. Check for patterns indicating assumption violations

In [ ]:
# Exercise 2 Solution

# Calculate residuals
y_train_pred = model_banking.predict(X_train_banking)
y_test_pred = model_banking.predict(X_test_banking)

residuals_train = y_train_banking.ravel() - y_train_pred.ravel()
residuals_test = y_test_banking.ravel() - y_test_pred.ravel()

print("=== RESIDUAL ANALYSIS ===")
print(f"Training residuals - Mean: {residuals_train.mean():.6f}, Std: {residuals_train.std():.4f}")
print(f"Test residuals - Mean: {residuals_test.mean():.6f}, Std: {residuals_test.std():.4f}")
print(f"Share of test residuals below -2: {np.mean(residuals_test < -2):.3f}; above +2: {np.mean(residuals_test > 2):.3f}")

# A 2 x 2 panel of diagnostic plots on the TEST residuals.
# (No residuals-vs-row-index plot: the rows are shuffled calls, so the index carries no information.)
from scipy import stats
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# 1. Residuals vs fitted values (homoscedasticity, and a check for curvature)
axes[0,0].scatter(y_test_pred, residuals_test, alpha=0.3, s=8)
axes[0,0].axhline(y=0, color='red', linestyle='--')
axes[0,0].set_xlabel('Fitted values')
axes[0,0].set_ylabel('Residuals')
axes[0,0].set_title('Residuals vs Fitted (Test)')

# 2. Histogram of the residuals with the normal density of the same mean and sd
axes[0,1].hist(residuals_test, bins=60, density=True, alpha=0.7)
grid = np.linspace(residuals_test.min(), residuals_test.max(), 200)
axes[0,1].plot(grid, stats.norm.pdf(grid, residuals_test.mean(), residuals_test.std()), 'r-', label='normal with same mean and sd')
axes[0,1].set_xlabel('Residuals')
axes[0,1].set_ylabel('Density')
axes[0,1].set_title('Distribution of Residuals (Test)')
axes[0,1].legend()

# 3. Q-Q plot against theoretical normal quantiles
stats.probplot(residuals_test, dist="norm", plot=axes[1,0])
axes[1,0].set_title('Normal Q-Q Plot (Test)')

# 4. Residuals vs one regressor (age): a pattern here would mean the model misses something in age
age_idx = list(X_df_banking.columns).index('age')
axes[1,1].scatter(X_test_banking[:, age_idx], residuals_test, alpha=0.3, s=8)
axes[1,1].axhline(y=0, color='red', linestyle='--')
axes[1,1].set_xlabel('Age')
axes[1,1].set_ylabel('Residuals')
axes[1,1].set_title('Residuals vs Age (Test)')

plt.tight_layout()
plt.show()


In [ ]:
# Statistical tests for normality
from scipy.stats import shapiro, jarque_bera, anderson

print("=== NORMALITY TESTS ===")

# Shapiro-Wilk test (good for smaller samples)
if len(residuals_test) <= 5000:  # Shapiro-Wilk has sample size limitations
    shapiro_stat, shapiro_p = shapiro(residuals_test)
    print(f"Shapiro-Wilk Test: statistic={shapiro_stat:.4f}, p-value={shapiro_p:.6f}")
    print(f"Interpretation: {'Residuals appear normal' if shapiro_p > 0.05 else 'Residuals deviate from normality'}")

# Jarque-Bera test
jb_stat, jb_p = jarque_bera(residuals_test)
print(f"\nJarque-Bera Test: statistic={jb_stat:.4f}, p-value={jb_p:.6f}")
print(f"Interpretation: {'Residuals appear normal' if jb_p > 0.05 else 'Residuals deviate from normality'}")

# Anderson-Darling test
ad_stat, ad_critical, ad_significance = anderson(residuals_test, dist='norm')
print(f"\nAnderson-Darling Test: statistic={ad_stat:.4f}")
for i, (crit, sig) in enumerate(zip(ad_critical, ad_significance)):
    print(f"  At {sig}% significance: critical value = {crit:.4f}, {'REJECT normality' if ad_stat > crit else 'ACCEPT normality'}")

### Residual Analysis Interpretation (from this run):

One sentence per panel:

1. **Residuals vs fitted (top left).** The cloud is centred on zero with the same vertical spread (about -2 to +2) across fitted values from 4.8 to 5.9, so there is no funnel and no curvature; the model is not systematically wrong for long or short predicted calls. The hard diagonal edge at the bottom left is the handful of calls with duration 0, whose residual is 0 minus the fitted value.
2. **Histogram (top right).** The residuals are roughly bell-shaped but skewed to the left: the left tail is fatter than the normal curve with the same mean and standard deviation, and the peak sits slightly to the right of zero. Very short calls (residuals below -2, about 2.8% of the test set) are more common than very long ones (above +2, about 0.9%).
3. **Normal Q-Q plot (bottom left).** The points follow the line between about -1.5 and +2 theoretical quantiles and bend below it on the left, the same fat left tail as the histogram; the right tail is slightly lighter than normal. The residuals are not normal, but the departure is in the tails only.
4. **Residuals vs age (bottom right).** No trend and no change in spread from age 20 to 60, so nothing in age is left unexplained; the few clients above 70 are too sparse to say anything.

**What follows.** Homoscedasticity and linearity look fine, so the coefficient estimates and predictions are not distorted. Normality fails in the left tail. With 8,238 test residuals this has no practical effect on confidence intervals, and it explains why the test RMSE (0.906) is dominated by a minority of very short calls that no pre-call feature predicts. The training and test residual statistics are almost identical (sd 0.910 vs 0.906), so there is no sign of overfitting.


## Exercise 3: Feature Engineering

**Task:** Create new features and see if they improve model performance.

### Solution Approach:
1. Create interaction terms between numerical features
2. Add polynomial (squared) terms
3. Train new model with engineered features
4. Compare performance with original model

In [ ]:
# Exercise 3 Solution

print("=== FEATURE ENGINEERING ===")

# Start with original features
X_engineered = X_df_banking.copy()
print(f"Original features: {X_engineered.shape[1]}")

# 1. Create interaction terms between key numerical features
numerical_features = ['age', 'previous', 'pdays_clean', 'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed']
available_numerical = [col for col in numerical_features if col in X_engineered.columns]

print(f"\nCreating interactions between: {available_numerical}")

# Create some meaningful interactions (not all combinations to avoid overfitting)
interaction_pairs = [
    ('age', 'previous'),  # Age and previous contacts
    ('emp_var_rate', 'cons_conf_idx'),  # Economic indicators
    ('cons_price_idx', 'euribor3m'),  # Economic indicators
]

for feat1, feat2 in interaction_pairs:
    if feat1 in X_engineered.columns and feat2 in X_engineered.columns:
        interaction_name = f'{feat1}_x_{feat2}'
        X_engineered[interaction_name] = X_engineered[feat1] * X_engineered[feat2]
        print(f"Created interaction: {interaction_name}")

# 2. Add polynomial (squared) terms for key numerical features
polynomial_features = ['age', 'previous', 'emp_var_rate', 'cons_conf_idx']
for feat in polynomial_features:
    if feat in X_engineered.columns:
        squared_name = f'{feat}_squared'
        X_engineered[squared_name] = X_engineered[feat] ** 2
        print(f"Created polynomial term: {squared_name}")

print(f"\nTotal features after engineering: {X_engineered.shape[1]}")
print(f"Added {X_engineered.shape[1] - X_df_banking.shape[1]} new features")

In [ ]:
# Split the engineered dataset
X_eng_train, X_eng_test, y_eng_train, y_eng_test = train_test_split(
    X_engineered.values, y_banking, test_size=0.2, random_state=0
)

# Train model with engineered features
model_engineered = LinearRegression()
model_engineered.fit(X_eng_train, y_eng_train)

# Make predictions
y_eng_train_pred = model_engineered.predict(X_eng_train)
y_eng_test_pred = model_engineered.predict(X_eng_test)

# Calculate performance metrics
print("=== MODEL COMPARISON ===")

# Original model performance
r2_orig_train = r2_score(y_train_banking, y_train_pred)
r2_orig_test = r2_score(y_test_banking, y_test_pred)
rmse_orig_train = np.sqrt(mean_squared_error(y_train_banking, y_train_pred))
rmse_orig_test = np.sqrt(mean_squared_error(y_test_banking, y_test_pred))

# Engineered model performance
r2_eng_train = r2_score(y_eng_train, y_eng_train_pred)
r2_eng_test = r2_score(y_eng_test, y_eng_test_pred)
rmse_eng_train = np.sqrt(mean_squared_error(y_eng_train, y_eng_train_pred))
rmse_eng_test = np.sqrt(mean_squared_error(y_eng_test, y_eng_test_pred))

comparison_df = pd.DataFrame({
    'Metric': ['R² Train', 'R² Test', 'RMSE Train', 'RMSE Test'],
    'Original Model': [r2_orig_train, r2_orig_test, rmse_orig_train, rmse_orig_test],
    'Engineered Model': [r2_eng_train, r2_eng_test, rmse_eng_train, rmse_eng_test],
    'Improvement': [
        r2_eng_train - r2_orig_train,
        r2_eng_test - r2_orig_test,
        rmse_orig_train - rmse_eng_train,  # Negative means worse (higher RMSE)
        rmse_orig_test - rmse_eng_test
    ]
})

print(comparison_df.to_string(index=False, float_format='%.6f'))

# Check for overfitting
print(f"\n=== OVERFITTING CHECK ===")
print(f"Original model - Train/Test R² gap: {r2_orig_train - r2_orig_test:.6f}")
print(f"Engineered model - Train/Test R² gap: {r2_eng_train - r2_eng_test:.6f}")
print(f"Gap increase: {(r2_eng_train - r2_eng_test) - (r2_orig_train - r2_orig_test):.6f}")

In [ ]:
# Visualize the most important new features
new_features = [col for col in X_engineered.columns if col not in X_df_banking.columns]
new_feature_indices = [X_engineered.columns.get_loc(col) for col in new_features]
new_feature_coefs = model_engineered.coef_[0][new_feature_indices]

new_coef_df = pd.DataFrame({
    'Feature': new_features,
    'Coefficient': new_feature_coefs,
    'Abs_Coefficient': np.abs(new_feature_coefs)
}).sort_values('Abs_Coefficient', ascending=False)

print("\n=== NEW ENGINEERED FEATURES IMPORTANCE ===")
print(new_coef_df.to_string(index=False))

# Plot new features
if len(new_features) > 0:
    plt.figure(figsize=(10, 6))
    colors = ['red' if x < 0 else 'blue' for x in new_coef_df['Coefficient']]
    plt.barh(range(len(new_coef_df)), new_coef_df['Coefficient'], color=colors, alpha=0.7)
    plt.yticks(range(len(new_coef_df)), new_coef_df['Feature'])
    plt.xlabel('Coefficient Value')
    plt.title('Coefficients of Engineered Features')
    plt.axvline(x=0, color='black', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

### Feature Engineering Insights (from this run):

**Performance impact.** In this run the seven engineered terms (three interactions, four squares) move the test R² from about 0.013 to about 0.022 and the test RMSE from 0.9065 to 0.9023. The direction is right and the relative gain looks large (R² almost doubles), but in absolute terms it is one percentage point of explained variance and an RMSE improvement of 0.004 on a scale of 0.9: the pre-call features simply do not carry much information about call length, and squaring or multiplying them does not create information that is not there.

**Overfitting risk.** The train/test R² gap stays tiny (about 0.005 for the original and 0.003 for the engineered model) because the sample is large (33,000 training rows) relative to 32 regressors. With many more engineered terms, or a much smaller sample, the gap would open up.

**Feature importance.** `cons_price_idx_x_euribor3m` gets by far the largest coefficient among the new terms (about 0.19), followed by `emp_var_rate_squared`; the age terms are essentially zero. So the small gain comes from the macro block entering non-linearly. Given the VIFs of that block, this should be read as "the macro environment matters in a non-linear way" rather than as separate effects of prices and rates.

**Business reading.** A fair comparison keeps the dummy columns in both models (it did here). The engineered model is marginally better and harder to explain; for a staffing forecast that works with averages, the simpler model is the better choice.


## Exercise 4: Cross-Validation

**Task:** Use cross-validation to get a more robust estimate of model performance.

### Solution Approach:
1. Implement 5-fold cross-validation
2. Compare with baseline model
3. Analyze stability of performance
4. Discuss implications

In [ ]:
# Exercise 4 Solution

from sklearn.model_selection import cross_val_score, KFold

print("=== CROSS-VALIDATION ANALYSIS ===")

# Set up cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# 1. Cross-validate original model
cv_scores_r2 = cross_val_score(model_banking, X_banking, y_banking.ravel(), 
                               cv=cv, scoring='r2')
cv_scores_rmse = -cross_val_score(model_banking, X_banking, y_banking.ravel(), 
                                  cv=cv, scoring='neg_root_mean_squared_error')

print("Original Model Cross-Validation Results:")
print(f"R² scores: {cv_scores_r2}")
print(f"R² mean: {cv_scores_r2.mean():.6f} ± {cv_scores_r2.std():.6f}")
print(f"RMSE scores: {cv_scores_rmse}")
print(f"RMSE mean: {cv_scores_rmse.mean():.6f} ± {cv_scores_rmse.std():.6f}")

# 2. Cross-validate engineered model
cv_scores_eng_r2 = cross_val_score(model_engineered, X_engineered.values, y_banking.ravel(), 
                                   cv=cv, scoring='r2')
cv_scores_eng_rmse = -cross_val_score(model_engineered, X_engineered.values, y_banking.ravel(), 
                                      cv=cv, scoring='neg_root_mean_squared_error')

print("\nEngineered Model Cross-Validation Results:")
print(f"R² scores: {cv_scores_eng_r2}")
print(f"R² mean: {cv_scores_eng_r2.mean():.6f} ± {cv_scores_eng_r2.std():.6f}")
print(f"RMSE scores: {cv_scores_eng_rmse}")
print(f"RMSE mean: {cv_scores_eng_rmse.mean():.6f} ± {cv_scores_eng_rmse.std():.6f}")

# 3. Baseline model (predict mean)
dummy_regressor = DummyRegressor(strategy='mean')
cv_scores_dummy_r2 = cross_val_score(dummy_regressor, X_banking, y_banking.ravel(), 
                                     cv=cv, scoring='r2')
cv_scores_dummy_rmse = -cross_val_score(dummy_regressor, X_banking, y_banking.ravel(), 
                                        cv=cv, scoring='neg_root_mean_squared_error')

print("\nBaseline Model (Mean Prediction) Cross-Validation Results:")
print(f"R² mean: {cv_scores_dummy_r2.mean():.6f} ± {cv_scores_dummy_r2.std():.6f}")
print(f"RMSE mean: {cv_scores_dummy_rmse.mean():.6f} ± {cv_scores_dummy_rmse.std():.6f}")

In [ ]:
# Visualize cross-validation results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# R² comparison
cv_data_r2 = [cv_scores_dummy_r2, cv_scores_r2, cv_scores_eng_r2]
labels = ['Baseline\n(Mean)', 'Original\nModel', 'Engineered\nModel']

ax1.boxplot(cv_data_r2, tick_labels=labels)
ax1.set_ylabel('R² Score')
ax1.set_title('Cross-Validation R² Comparison')
ax1.grid(True, alpha=0.3)

# RMSE comparison
cv_data_rmse = [cv_scores_dummy_rmse, cv_scores_rmse, cv_scores_eng_rmse]

ax2.boxplot(cv_data_rmse, tick_labels=labels)
ax2.set_ylabel('RMSE')
ax2.set_title('Cross-Validation RMSE Comparison')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compare models by mean and spread across folds. We do NOT run a t-test on the fold scores:
# with n = 5 folds that share most of their training data, the scores are not independent draws,
# and the test would be p-value theatre rather than evidence.
print("\n=== MEAN +/- SD OF R² ACROSS FOLDS ===")
print(f"Baseline (mean):  {cv_scores_dummy_r2.mean():.4f} +/- {cv_scores_dummy_r2.std():.4f}")
print(f"Original model:   {cv_scores_r2.mean():.4f} +/- {cv_scores_r2.std():.4f}")
print(f"Engineered model: {cv_scores_eng_r2.mean():.4f} +/- {cv_scores_eng_r2.std():.4f}")
print(f"\nOriginal minus baseline, fold by fold: {np.round(cv_scores_r2 - cv_scores_dummy_r2, 4)}")
print(f"Engineered minus original, fold by fold: {np.round(cv_scores_eng_r2 - cv_scores_r2, 4)}")


In [ ]:
# Model stability analysis
print("\n=== MODEL STABILITY ANALYSIS ===")

def coefficient_of_variation(scores):
    return scores.std() / scores.mean() if scores.mean() != 0 else np.inf

stability_df = pd.DataFrame({
    'Model': ['Baseline', 'Original', 'Engineered'],
    'R² CV': [coefficient_of_variation(cv_scores_dummy_r2),
              coefficient_of_variation(cv_scores_r2),
              coefficient_of_variation(cv_scores_eng_r2)],
    'RMSE CV': [coefficient_of_variation(cv_scores_dummy_rmse),
                coefficient_of_variation(cv_scores_rmse),
                coefficient_of_variation(cv_scores_eng_rmse)]
})

print("Coefficient of Variation (lower = more stable):")
print(stability_df.to_string(index=False, float_format='%.6f'))

# Performance summary
summary_df = pd.DataFrame({
    'Model': ['Baseline', 'Original', 'Engineered'],
    'Mean R²': [cv_scores_dummy_r2.mean(), cv_scores_r2.mean(), cv_scores_eng_r2.mean()],
    'Std R²': [cv_scores_dummy_r2.std(), cv_scores_r2.std(), cv_scores_eng_r2.std()],
    'Mean RMSE': [cv_scores_dummy_rmse.mean(), cv_scores_rmse.mean(), cv_scores_eng_rmse.mean()],
    'Std RMSE': [cv_scores_dummy_rmse.std(), cv_scores_rmse.std(), cv_scores_eng_rmse.std()]
})

print("\nPerformance Summary:")
print(summary_df.to_string(index=False, float_format='%.6f'))

### Cross-Validation Insights (from this run):

**Robust performance estimates.** The five-fold R² of the original model is about 0.016 +/- 0.003 in this run, against 0.000 for the mean-only baseline (a baseline fitted on the training folds has R² slightly below zero on the held-out fold, as expected). The single 80/20 split of the main notebook gave 0.013, inside the range of the folds. The engineered model reaches about 0.023 +/- 0.004.

**Stability.** The fold-to-fold standard deviation is a fifth of the mean, so the ranking baseline < original < engineered is the same on every fold (see the fold-by-fold differences printed above), even though the absolute level is tiny. The model is stable; it is just not powerful.

**Why no significance test.** A paired t-test on five fold scores is not appropriate: the folds overlap in 75% of their training data, so the scores are strongly dependent, and with n = 5 the t distribution is a poor approximation anyway. Reporting the mean, the spread and the fold-by-fold differences says everything a test would, without pretending to a p-value.

**Business implication.** Cross-validation confirms that the 1 to 2% explained variance is a property of the data, not of one lucky or unlucky split. A call centre should not expect a per-call duration forecast from these features.


## Exercise 5: Synthetic Data Generation

**Task:** Create your own synthetic dataset with known relationships and test your model.

### Solution Approach:
1. Generate synthetic data with known coefficients
2. Add realistic noise and outliers
3. Test model's ability to recover true coefficients
4. Experiment with different noise levels

In [ ]:
# Exercise 5 Solution

print("=== SYNTHETIC DATA GENERATION ===")

# Set parameters
n_samples = 500
true_coefficients = np.array([2.0, 3.0, -1.5])  # Known true coefficients
true_intercept = 1.0

def generate_synthetic_data(n_samples, true_coef, true_intercept, noise_level=1.0, outlier_fraction=0.05):
    """
    Generate synthetic data with known linear relationship
    """
    # Generate features from different distributions to make it realistic
    X1 = np.random.normal(0, 1, n_samples)  # Standard normal
    X2 = np.random.uniform(-2, 2, n_samples)  # Uniform
    X3 = np.random.exponential(1, n_samples)  # Exponential (right-skewed)
    
    X = np.column_stack([X1, X2, X3])
    
    # Generate target with known relationship
    y_true = true_intercept + X @ true_coef
    
    # Add noise
    noise = np.random.normal(0, noise_level, n_samples)
    y = y_true + noise
    
    # Add outliers
    n_outliers = int(outlier_fraction * n_samples)
    outlier_indices = np.random.choice(n_samples, n_outliers, replace=False)
    y[outlier_indices] += np.random.normal(0, 5 * noise_level, n_outliers)
    
    return X, y, y_true

# Generate synthetic data with different noise levels
noise_levels = [0.5, 1.0, 2.0, 3.0]
results = []

for noise_level in noise_levels:
    print(f"\n--- Noise Level: {noise_level} ---")
    
    # Generate data
    X_syn, y_syn, y_true_syn = generate_synthetic_data(
        n_samples, true_coefficients, true_intercept, noise_level
    )
    
    # Split data
    X_train_syn, X_test_syn, y_train_syn, y_test_syn = train_test_split(
        X_syn, y_syn, test_size=0.2, random_state=42
    )
    
    # Train model
    model_syn = LinearRegression()
    model_syn.fit(X_train_syn, y_train_syn)
    
    # Evaluate
    y_pred_syn = model_syn.predict(X_test_syn)
    r2_syn = r2_score(y_test_syn, y_pred_syn)
    rmse_syn = np.sqrt(mean_squared_error(y_test_syn, y_pred_syn))
    
    # Compare estimated vs true coefficients
    estimated_coef = model_syn.coef_
    estimated_intercept = model_syn.intercept_
    
    coef_error = np.abs(estimated_coef - true_coefficients)
    intercept_error = abs(estimated_intercept - true_intercept)
    
    print(f"True coefficients: {true_coefficients}")
    print(f"Estimated coefficients: {estimated_coef}")
    print(f"Coefficient errors: {coef_error}")
    print(f"True intercept: {true_intercept:.3f}, Estimated: {estimated_intercept:.3f}, Error: {intercept_error:.3f}")
    print(f"R²: {r2_syn:.6f}, RMSE: {rmse_syn:.6f}")
    
    results.append({
        'noise_level': noise_level,
        'r2': r2_syn,
        'rmse': rmse_syn,
        'coef_error_mean': coef_error.mean(),
        'coef_error_max': coef_error.max(),
        'intercept_error': intercept_error
    })

In [ ]:
# Visualize results
results_df = pd.DataFrame(results)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# R² vs noise level
axes[0,0].plot(results_df['noise_level'], results_df['r2'], 'bo-')
axes[0,0].set_xlabel('Noise Level')
axes[0,0].set_ylabel('R²')
axes[0,0].set_title('R² vs Noise Level')
axes[0,0].grid(True, alpha=0.3)

# RMSE vs noise level
axes[0,1].plot(results_df['noise_level'], results_df['rmse'], 'ro-')
axes[0,1].set_xlabel('Noise Level')
axes[0,1].set_ylabel('RMSE')
axes[0,1].set_title('RMSE vs Noise Level')
axes[0,1].grid(True, alpha=0.3)

# Coefficient error vs noise level
axes[1,0].plot(results_df['noise_level'], results_df['coef_error_mean'], 'go-', label='Mean Error')
axes[1,0].plot(results_df['noise_level'], results_df['coef_error_max'], 'g^-', label='Max Error')
axes[1,0].set_xlabel('Noise Level')
axes[1,0].set_ylabel('Coefficient Error')
axes[1,0].set_title('Coefficient Recovery Error vs Noise Level')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Intercept error vs noise level
axes[1,1].plot(results_df['noise_level'], results_df['intercept_error'], 'mo-')
axes[1,1].set_xlabel('Noise Level')
axes[1,1].set_ylabel('Intercept Error')
axes[1,1].set_title('Intercept Recovery Error vs Noise Level')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== SUMMARY TABLE ===")
print(results_df.to_string(index=False, float_format='%.6f'))

In [ ]:
# Demonstrate with a specific example (medium noise)
X_demo, y_demo, y_true_demo = generate_synthetic_data(
    n_samples, true_coefficients, true_intercept, noise_level=1.0
)

# Visualize the data
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

feature_names = ['X1 (Normal)', 'X2 (Uniform)', 'X3 (Exponential)']

for i in range(3):
    axes[i].scatter(X_demo[:, i], y_demo, alpha=0.6, label='Observed')
    axes[i].scatter(X_demo[:, i], y_true_demo, alpha=0.6, color='red', s=10, label='True (no noise)')
    axes[i].set_xlabel(feature_names[i])
    axes[i].set_ylabel('Target')
    axes[i].set_title(f'Target vs {feature_names[i]}')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Train final model and show detailed results
model_demo = LinearRegression()
model_demo.fit(X_demo, y_demo)

print("\n=== DETAILED COEFFICIENT RECOVERY ===")
coef_comparison = pd.DataFrame({
    'Feature': ['X1', 'X2', 'X3', 'Intercept'],
    'True Value': list(true_coefficients) + [true_intercept],
    'Estimated Value': list(model_demo.coef_) + [model_demo.intercept_],
    'Absolute Error': list(np.abs(model_demo.coef_ - true_coefficients)) + [abs(model_demo.intercept_ - true_intercept)],
    'Relative Error (%)': [
        abs(est - true) / abs(true) * 100 if true != 0 else np.inf
        for est, true in zip(list(model_demo.coef_) + [model_demo.intercept_], 
                           list(true_coefficients) + [true_intercept])
    ]
})

print(coef_comparison.to_string(index=False, float_format='%.6f'))

### Synthetic Data Insights (from this run):

**What you should have predicted.** Adding noise to the target does not bias ordinary least squares, so the coefficient estimates should stay centred on (2, 3, -1.5) at every noise level, but they should become less precise (larger errors from run to run), and R² must fall because a growing share of the variation in y is noise that no model can explain.

**What happened.** That is what the summary table shows. The mean absolute coefficient error grows from 0.02 at noise 0.5 to 0.08 at noise 1, 0.15 at noise 2 and 0.16 at noise 3, with the largest single error 0.30 (X1 at noise 3, estimated 1.70 instead of 2); the errors are of both signs, as unbiased estimates should be. R² falls from 0.97 to 0.84 to 0.62. That the R² at noise 3 (0.624) is not below the R² at noise 2 (0.619) is sampling variation: each noise level draws a fresh 500-observation dataset, and the 5% of outliers, which get a five times larger shock, move R² by a few hundredths on their own.

**The demonstration run (noise 1, with outliers).** The three slopes are recovered within 0.07 (0.3%, 0.8% and 4.4% relative error), but the intercept comes out at 0.83 instead of 1.00, a 17% error. The outliers were added to y with mean zero, so in expectation they do not shift the intercept, but 25 outliers with a standard deviation of 5 can easily move the sample mean of y by 0.2; the slopes, which depend on the covariance with x, are more robust to that.

**Why this matters.** Knowing the truth lets you check that the estimator behaves as the theory says, and it calibrates your expectations for real data: on the banking model there is no truth to compare with, but the same logic says that a low R² (much noise) does not by itself make the coefficient estimates wrong, only less precise.


## Exercise 6: Time Series Forecasting Analysis

**Task:** Analyze the Bitcoin forecasting model more deeply.

### Solution Approach:
1. Calculate directional accuracy
2. Create cumulative returns comparison
3. Test different lag lengths
4. Discuss trading implications

In [ ]:
# Exercise 6 Solution
# First, let's recreate the Bitcoin data from the main notebook

print("=== BITCOIN TIME SERIES ANALYSIS ===")

# Load Bitcoin data: one row per day, columns Date and BTC-USD.Close
btc_url = "https://raw.githubusercontent.com/umatter/EDFB/main/data/data_BTC.csv"
data_btc = pd.read_csv(btc_url)
data_btc['Date'] = pd.to_datetime(data_btc['Date'])
data_btc = data_btc.sort_values('Date').reset_index(drop=True)

print(f"Bitcoin data loaded: {len(data_btc)} observations")
print(f"Date range: {data_btc['Date'].min().date()} to {data_btc['Date'].max().date()}")


In [ ]:
# Function to create lagged features and train the model.
# Three chronological slices: train (first 60% of days), validation (next 20%), test (last 20%).
# The lag length is CHOSEN on the validation slice and only the chosen model is judged on the test slice;
# choosing on the test slice would be the data snooping that Exercise 4 warns about.
def create_lagged_features_btc(data, lag):
    df = data.copy()
    df['ret'] = np.log(df['BTC-USD.Close']).diff()
    for i in range(1, lag+1):
        df[f'lag_ret_{i}'] = df['ret'].shift(i)
    df = df.dropna().reset_index(drop=True)
    return df

def train_btc_model(data, lag):
    data_lagged = create_lagged_features_btc(data, lag)
    feature_cols = [f'lag_ret_{i}' for i in range(1, lag+1)]   # lagged returns only: nothing observed on day t
    X = data_lagged[feature_cols].values
    y = data_lagged['ret'].values

    n = len(data_lagged)
    test_start = int(n * 0.8)          # same test period as the main notebook (last 20%)
    val_start = int(test_start * 0.75) # last quarter of the training period is the validation slice

    X_fit, y_fit = X[:val_start], y[:val_start]
    X_val, y_val = X[val_start:test_start], y[val_start:test_start]
    X_train, y_train = X[:test_start], y[:test_start]
    X_test, y_test = X[test_start:], y[test_start:]

    # Model used for selection: fitted on the first 60%, judged on the validation slice
    model_val = LinearRegression().fit(X_fit, y_fit)
    y_val_pred = model_val.predict(X_val)
    # Model used for the final test: refitted on the whole training period (80%)
    model = LinearRegression().fit(X_train, y_train)
    y_pred = model.predict(X_test)

    return {
        'model': model, 'X_train': X_train, 'y_train': y_train,
        'X_test': X_test, 'y_test': y_test, 'y_pred': y_pred,
        'y_val': y_val, 'y_val_pred': y_val_pred,
        'data': data_lagged, 'test_start': test_start,
        'dates_test': data_lagged['Date'].values[test_start:],
    }

def directional_accuracy(y_true, y_pred):
    return np.mean(np.sign(y_true) == np.sign(y_pred))

# Test different lag lengths
lag_lengths = [1, 3, 5, 10]
lag_results = {}

print("\n=== TESTING DIFFERENT LAG LENGTHS (selection metrics on the VALIDATION slice) ===")

for lag in lag_lengths:
    result = train_btc_model(data_btc, lag)
    lag_results[lag] = result

    # Validation metrics (the only metrics used for selection). Benchmark: the zero forecast.
    # The test slice is not looked at here: it is used once, for the chosen lag, in the next cells.
    rmse_val = np.sqrt(mean_squared_error(result['y_val'], result['y_val_pred']))
    rmse_val_zero = np.sqrt(np.mean(result['y_val']**2))
    da_val = directional_accuracy(result['y_val'], result['y_val_pred'])
    up_val = np.mean(result['y_val'] > 0)

    print(f"\n--- Lag Length: {lag} ---")
    print(f"Validation: RMSE ratio (model/zero) = {rmse_val/rmse_val_zero:.4f}, directional accuracy = {da_val:.3f} (share of up-days {up_val:.3f})")

    lag_results[lag]['metrics'] = {
        'val_rmse_ratio': rmse_val/rmse_val_zero, 'val_directional_accuracy': da_val, 'val_up_share': up_val,
    }


In [ ]:
# Compare lag lengths
comparison_data = []
for lag, result in lag_results.items():
    m = result['metrics']
    comparison_data.append({
        'Lag Length': lag,
        'Val RMSE Ratio': m['val_rmse_ratio'],
        'Val Dir. Accuracy': m['val_directional_accuracy'],
        'Val Up-day Share': m['val_up_share'],
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n=== LAG LENGTH COMPARISON ON THE VALIDATION SLICE (RMSE ratio is model / zero forecast; below 1 = skill) ===")
print(comparison_df.to_string(index=False, float_format='%.4f'))

# Choose the lag on the VALIDATION slice only
best_lag = int(comparison_df.loc[comparison_df['Val Dir. Accuracy'].idxmax(), 'Lag Length'])
print(f"\nLag chosen on the validation slice (highest validation directional accuracy): {best_lag}")

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(comparison_df['Lag Length'], comparison_df['Val RMSE Ratio'], 'bo-', label='Validation')
axes[0].axhline(y=1, color='black', linestyle='--', alpha=0.5, label='Zero forecast')
axes[0].set_xlabel('Lag Length')
axes[0].set_ylabel('RMSE Ratio (Model / Zero forecast)')
axes[0].set_title('Validation RMSE Ratio vs Lag Length')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(comparison_df['Lag Length'], comparison_df['Val Dir. Accuracy'], 'bo-', label='Validation')
axes[1].plot(comparison_df['Lag Length'], comparison_df['Val Up-day Share'], 'k:', label='Validation share of up-days ("always up")')
axes[1].set_xlabel('Lag Length')
axes[1].set_ylabel('Directional Accuracy')
axes[1].set_title('Validation Directional Accuracy vs Lag Length')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# The one look at the TEST slice: the model with the lag chosen on the validation slice, refitted on the whole training period
print(f"\n=== TEST RESULT OF THE CHOSEN MODEL (Lag {best_lag}, test period) ===")

best_result = lag_results[best_lag]
y_test = best_result['y_test']
y_pred = best_result['y_pred']
n_test = len(y_test)

rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
rmse_test_zero = np.sqrt(np.mean(y_test**2))
r2_test = r2_score(y_test, y_pred)
da_test = directional_accuracy(y_test, y_pred)
up_share = np.mean(y_test > 0)
se_da = np.sqrt(0.25 / n_test)   # standard error of an accuracy estimate near 0.5 with n_test days
print(f"RMSE ratio (model / zero forecast): {rmse_test/rmse_test_zero:.4f}   (below 1 would mean forecasting skill)")
print(f"R-squared on the test set: {r2_test:.4f}")
print(f"Directional accuracy on the test set: {da_test:.3f} ({int(round(da_test*n_test))}/{n_test} days)")
print(f"Share of up-days (accuracy of 'always predict up'): {up_share:.3f}")
print(f"Standard error of an accuracy estimate with {n_test} days: about {se_da:.3f}")

print(f"\n=== CUMULATIVE RETURNS ANALYSIS (Lag {best_lag}, test period) ===")

# Trading strategies (positions decided at the end of day t-1, earning the log return of day t)
# Strategy 1: perfect foresight: long when the return is positive, short when it is negative (earns |r_t| every day)
perfect_returns = np.abs(y_test)
# Strategy 2: follow the model's sign: long if the predicted return is positive, short otherwise
position = np.where(y_pred > 0, 1, -1)
model_returns = position * y_test
# Strategy 2b: the same, after a transaction cost of 0.5% (50 basis points) each time the position changes
cost_per_switch = 0.005
switches = np.concatenate([[1], (np.diff(position) != 0).astype(int)])   # the first day counts as entering a position
model_returns_net = model_returns - cost_per_switch * switches
# Strategy 3: buy and hold (always long)
buy_hold_returns = y_test
# Strategy 4: random long/short
np.random.seed(42)
random_signals = np.random.choice([-1, 1], size=n_test)
random_returns = random_signals * y_test

strategies = {
    'Perfect foresight': perfect_returns,
    f'Model sign (lag {best_lag}), no costs': model_returns,
    f'Model sign (lag {best_lag}), 50 bp per switch': model_returns_net,
    'Buy and hold': buy_hold_returns,
    'Random': random_returns,
}

# Plot cumulative log returns
plt.figure(figsize=(15, 8))
for name, r in strategies.items():
    plt.plot(best_result['dates_test'], np.cumsum(r), label=name, linewidth=2 if 'Model' in name else 1)
plt.xlabel('Date')
plt.ylabel('Cumulative log return')
plt.title('Cumulative returns of trading strategies on the test period')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Strategy statistics (per day, not annualised)
strategy_stats = []
for name, r in strategies.items():
    cum = np.cumsum(r)
    strategy_stats.append({
        'Strategy': name,
        'Total log return': np.sum(r),
        'Mean daily return': np.mean(r),
        'Daily volatility': np.std(r),
        'Sharpe (daily, unannualised)': np.mean(r) / np.std(r) if np.std(r) > 0 else 0,
        'Max drawdown': np.min(cum - np.maximum.accumulate(cum)),
    })
strategy_df = pd.DataFrame(strategy_stats)
print(f"\nNumber of position switches by the model strategy: {int(switches.sum())} in {n_test} days")
print("\nStrategy performance on the test period:")
print(strategy_df.to_string(index=False, float_format='%.4f'))


In [ ]:
# Detailed analysis of model predictions
print("\n=== DETAILED PREDICTION ANALYSIS ===")

# Prediction accuracy by magnitude
pred_magnitude = np.abs(y_pred)
actual_magnitude = np.abs(y_test)

# Bin predictions by confidence (magnitude)
confidence_bins = np.percentile(pred_magnitude, [0, 25, 50, 75, 100])
bin_labels = ['Low', 'Medium-Low', 'Medium-High', 'High']

print("Directional Accuracy by Prediction Confidence:")
for i, label in enumerate(bin_labels):
    mask = (pred_magnitude >= confidence_bins[i]) & (pred_magnitude < confidence_bins[i+1])
    if i == len(bin_labels) - 1:  # Include the maximum value in the last bin
        mask = pred_magnitude >= confidence_bins[i]
    
    if np.sum(mask) > 0:
        accuracy = np.mean(np.sign(y_test[mask]) == np.sign(y_pred[mask]))
        count = np.sum(mask)
        avg_magnitude = np.mean(pred_magnitude[mask])
        print(f"{label} Confidence: {accuracy:.3f} ({count} predictions, avg magnitude: {avg_magnitude:.6f})")

# Market regime analysis
print("\nDirectional Accuracy by Market Regime:")

# Define regimes based on actual return magnitude
low_vol_mask = actual_magnitude < np.percentile(actual_magnitude, 33)
med_vol_mask = (actual_magnitude >= np.percentile(actual_magnitude, 33)) & (actual_magnitude < np.percentile(actual_magnitude, 67))
high_vol_mask = actual_magnitude >= np.percentile(actual_magnitude, 67)

regimes = {
    'Low Volatility': low_vol_mask,
    'Medium Volatility': med_vol_mask,
    'High Volatility': high_vol_mask
}

for regime_name, mask in regimes.items():
    if np.sum(mask) > 0:
        accuracy = np.mean(np.sign(y_test[mask]) == np.sign(y_pred[mask]))
        count = np.sum(mask)
        avg_return = np.mean(np.abs(y_test[mask]))
        print(f"{regime_name}: {accuracy:.3f} ({count} periods, avg |return|: {avg_return:.6f})")

### Time Series Forecasting Insights (from this run):

**Lag length.** On the validation slice (the last quarter of the training period) the four lag lengths reach directional accuracies of about 0.52 to 0.55 (against a validation share of up-days of 0.508) and RMSE ratios of 0.994 to 0.997, so lag 10 was chosen. The test slice was then used once, for that model only. Its RMSE ratio against the zero forecast is 1.005 and its test R² is about -0.010: it does not beat "no change". Its directional accuracy is about 0.517 against a share of up-days of about 0.494: 15 more correct days out of 662 than "always up", with a standard error of about 0.019. That is inside the noise. The small validation edge (0.55 against 0.51) did not carry over, which is the usual fate of a pattern selected from a handful of candidates on noisy returns.

**Directional accuracy is not a free lunch.** 0.5 is not the right yardstick; the share of up-days is (0.494 here). A strategy that predicts "up" every day gets that accuracy with no model at all.

**Trading.** Before costs the model-sign strategy ends the test period almost exactly flat (cumulative log return of about 0.01), which happens to beat buy and hold (about -0.27, Bitcoin fell over this test period) and the random strategy (about -0.50), but its daily Sharpe ratio is 0.001: indistinguishable from zero. It switches position on about 43% of the days (283 switches in 662 days); at 50 basis points per switch the costs sum to about 1.4 in log-return terms, a hundred times the gross gain, and the net line ends at about -1.4. The "perfect foresight" line (earning |r| every day, about +10.7) shows how much is theoretically there and how little of it a linear model on lagged returns captures.

**Conclusion.** Linear models on lagged daily returns have no usable predictive power for Bitcoin in this data. This is the weak-form efficient-market result: past returns are already in the price. The main notebook's Durbin-Watson and ACF checks said the same thing from the other side: the residuals, and the returns themselves, are white noise. "Improving" the model with more lags moves accuracy by a percentage point or two, well within the standard error, and costs turn even a genuine edge of that size into a loss.


## Exercise 7: Model Comparison

**Task:** Compare linear regression with Ridge and Lasso regression.

### Solution Approach:
1. Implement Ridge and Lasso regression
2. Tune the regularisation strength by cross-validation on the training data
3. Compare performance metrics on the test set
4. Analyse the feature-selection effect of Lasso

This exercise uses the engineered 32-column matrix of Exercise 3 (interactions and squares included, so that Lasso has something to select) and the same 80/20 split seed as the main notebook and Exercise 3. The "Linear Regression" row is therefore the engineered model of Exercise 3 (test R² about 0.022 in this run), not the 25-regressor main model (about 0.013).


In [ ]:
# Exercise 7 Solution

from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

print("=== REGULARIZED REGRESSION COMPARISON ===")

# Use the banking dataset with engineered features for this comparison
X_comparison = X_engineered.values
y_comparison = y_banking.ravel()

# Split the data
X_train_comp, X_test_comp, y_train_comp, y_test_comp = train_test_split(
    X_comparison, y_comparison, test_size=0.2, random_state=0     # same split as the main notebook and Exercise 3
)

# Standardize features (important for regularized regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_comp)
X_test_scaled = scaler.transform(X_test_comp)

print(f"Dataset: {X_train_scaled.shape[0]} training, {X_test_scaled.shape[0]} test samples")
print(f"Features: {X_train_scaled.shape[1]}")

# Define models and hyperparameter grids
models = {
    'Linear Regression': {
        'model': LinearRegression(),
        'params': {}
    },
    'Ridge Regression': {
        'model': Ridge(),
        'params': {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}
    },
    'Lasso Regression': {
        'model': Lasso(max_iter=2000),
        'params': {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}
    }
}

# Train and evaluate models
results = {}

for name, config in models.items():
    print(f"\n--- Training {name} ---")
    
    if config['params']:  # If hyperparameters to tune
        # Use GridSearchCV for hyperparameter tuning
        grid_search = GridSearchCV(
            config['model'], config['params'], 
            cv=5, scoring='r2', n_jobs=-1
        )
        grid_search.fit(X_train_scaled, y_train_comp)
        best_model = grid_search.best_estimator_
        best_params = grid_search.best_params_
        print(f"Best parameters: {best_params}")
        print(f"Best CV score: {grid_search.best_score_:.6f}")
    else:
        # No hyperparameters to tune
        best_model = config['model']
        best_model.fit(X_train_scaled, y_train_comp)
        best_params = {}
    
    # Make predictions
    y_train_pred = best_model.predict(X_train_scaled)
    y_test_pred = best_model.predict(X_test_scaled)
    
    # Calculate metrics
    train_r2 = r2_score(y_train_comp, y_train_pred)
    test_r2 = r2_score(y_test_comp, y_test_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train_comp, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test_comp, y_test_pred))
    
    # Store results
    results[name] = {
        'model': best_model,
        'params': best_params,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'overfitting': train_r2 - test_r2
    }
    
    print(f"Train R²: {train_r2:.6f}, Test R²: {test_r2:.6f}")
    print(f"Train RMSE: {train_rmse:.6f}, Test RMSE: {test_rmse:.6f}")
    print(f"Overfitting (R² gap): {train_r2 - test_r2:.6f}")

In [ ]:
# Create comparison table
comparison_data = []
for name, result in results.items():
    comparison_data.append({
        'Model': name,
        'Train R²': result['train_r2'],
        'Test R²': result['test_r2'],
        'Train RMSE': result['train_rmse'],
        'Test RMSE': result['test_rmse'],
        'Overfitting': result['overfitting'],
        'Best Params': str(result['params'])
    })

comparison_table = pd.DataFrame(comparison_data)
print("\n=== MODEL COMPARISON SUMMARY ===")
print(comparison_table.to_string(index=False, float_format='%.6f'))

# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

models_list = list(results.keys())
train_r2_values = [results[model]['train_r2'] for model in models_list]
test_r2_values = [results[model]['test_r2'] for model in models_list]
train_rmse_values = [results[model]['train_rmse'] for model in models_list]
test_rmse_values = [results[model]['test_rmse'] for model in models_list]

# R² comparison
x_pos = np.arange(len(models_list))
width = 0.35

axes[0,0].bar(x_pos - width/2, train_r2_values, width, label='Train', alpha=0.8)
axes[0,0].bar(x_pos + width/2, test_r2_values, width, label='Test', alpha=0.8)
axes[0,0].set_xlabel('Model')
axes[0,0].set_ylabel('R²')
axes[0,0].set_title('R² Comparison')
axes[0,0].set_xticks(x_pos)
axes[0,0].set_xticklabels(models_list, rotation=45)
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# RMSE comparison
axes[0,1].bar(x_pos - width/2, train_rmse_values, width, label='Train', alpha=0.8)
axes[0,1].bar(x_pos + width/2, test_rmse_values, width, label='Test', alpha=0.8)
axes[0,1].set_xlabel('Model')
axes[0,1].set_ylabel('RMSE')
axes[0,1].set_title('RMSE Comparison')
axes[0,1].set_xticks(x_pos)
axes[0,1].set_xticklabels(models_list, rotation=45)
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Overfitting comparison
overfitting_values = [results[model]['overfitting'] for model in models_list]
axes[1,0].bar(models_list, overfitting_values, alpha=0.8, color='red')
axes[1,0].set_xlabel('Model')
axes[1,0].set_ylabel('Overfitting (Train R² - Test R²)')
axes[1,0].set_title('Overfitting Comparison')
axes[1,0].tick_params(axis='x', rotation=45)
axes[1,0].grid(True, alpha=0.3)

# Test R² vs Overfitting scatter
axes[1,1].scatter(test_r2_values, overfitting_values, s=100, alpha=0.7)
for i, model in enumerate(models_list):
    axes[1,1].annotate(model, (test_r2_values[i], overfitting_values[i]), 
                      xytext=(5, 5), textcoords='offset points')
axes[1,1].set_xlabel('Test R²')
axes[1,1].set_ylabel('Overfitting')
axes[1,1].set_title('Test Performance vs Overfitting')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze feature selection (Lasso) and regularization effects
print("\n=== FEATURE SELECTION ANALYSIS ===")

# Get coefficients from each model
feature_names = X_engineered.columns

coef_comparison = pd.DataFrame({
    'Feature': feature_names,
    'Linear': results['Linear Regression']['model'].coef_,
    'Ridge': results['Ridge Regression']['model'].coef_,
    'Lasso': results['Lasso Regression']['model'].coef_
})

# Count non-zero coefficients
print("Non-zero coefficients:")
for model in ['Linear', 'Ridge', 'Lasso']:
    non_zero = np.sum(np.abs(coef_comparison[model]) > 1e-6)
    print(f"{model}: {non_zero}/{len(feature_names)} ({non_zero/len(feature_names)*100:.1f}%)")

# Show features selected by Lasso
lasso_selected = coef_comparison[np.abs(coef_comparison['Lasso']) > 1e-6]
print(f"\nFeatures selected by Lasso ({len(lasso_selected)}):")
lasso_selected_sorted = lasso_selected.reindex(
    lasso_selected['Lasso'].abs().sort_values(ascending=False).index
)
print(lasso_selected_sorted[['Feature', 'Lasso']].to_string(index=False, float_format='%.6f'))

# Visualize coefficient comparison for top features
top_features = coef_comparison.reindex(
    coef_comparison['Linear'].abs().sort_values(ascending=False).index
).head(15)

fig, ax = plt.subplots(figsize=(12, 8))
x_pos = np.arange(len(top_features))
width = 0.25

ax.bar(x_pos - width, top_features['Linear'], width, label='Linear', alpha=0.8)
ax.bar(x_pos, top_features['Ridge'], width, label='Ridge', alpha=0.8)
ax.bar(x_pos + width, top_features['Lasso'], width, label='Lasso', alpha=0.8)

ax.set_xlabel('Features')
ax.set_ylabel('Coefficient Value (standardised features)')
ax.set_title('Coefficient Comparison: Top 15 Features')
ax.set_xticks(x_pos)
ax.set_xticklabels(top_features['Feature'], rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Regularization path analysis.
# The regularisation strength is chosen by 5-fold cross-validation on the TRAINING data only.
# Scoring the path on the test set and picking the best alpha would tune the model to the test set,
# and the reported test R² would then be optimistic.
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import lasso_path

cv_path = KFold(n_splits=5, shuffle=True, random_state=42)
print("\n=== REGULARIZATION PATH ANALYSIS (CV on training data) ===")

# Ridge regularization path (Ridge has a closed-form solution, so refitting per alpha is cheap)
alphas_ridge = np.logspace(-3, 3, 20)
ridge_coefs, ridge_cv_scores = [], []
for alpha in alphas_ridge:
    ridge = Ridge(alpha=alpha).fit(X_train_scaled, y_train_comp)
    ridge_coefs.append(ridge.coef_)
    ridge_cv_scores.append(cross_val_score(Ridge(alpha=alpha), X_train_scaled, y_train_comp, cv=cv_path, scoring='r2').mean())
ridge_coefs = np.array(ridge_coefs)

# Lasso regularization path. Refitting Lasso from scratch for every alpha is very slow at small alphas;
# lasso_path walks down the alphas with warm starts (the whole path takes well under a second).
# lasso_path fits no intercept, so centre X and y and add the mean back when predicting.
alphas_lasso = np.logspace(-4, 0, 20)

def lasso_cv_r2(alphas):
    scores = np.zeros((len(alphas), cv_path.get_n_splits()))
    for k, (fit_idx, val_idx) in enumerate(cv_path.split(X_train_scaled)):
        X_fit, X_val = X_train_scaled[fit_idx], X_train_scaled[val_idx]
        y_fit, y_val = y_train_comp[fit_idx], y_train_comp[val_idx]
        x_mean, y_mean = X_fit.mean(axis=0), y_fit.mean()
        alphas_out, coefs, _ = lasso_path(X_fit - x_mean, y_fit - y_mean, alphas=alphas, max_iter=5000)
        preds = (X_val - x_mean) @ coefs + y_mean            # one column of predictions per alpha
        scores[:, k] = 1 - ((y_val[:, None] - preds)**2).sum(axis=0) / ((y_val - y_val.mean())**2).sum()
    return alphas_out, scores.mean(axis=1)                    # alphas come back in descending order

alphas_lasso, lasso_cv_scores = lasso_cv_r2(alphas_lasso)
x_mean_all, y_mean_all = X_train_scaled.mean(axis=0), y_train_comp.mean()
_, lasso_coefs, _ = lasso_path(X_train_scaled - x_mean_all, y_train_comp - y_mean_all, alphas=alphas_lasso, max_iter=5000)
lasso_coefs = lasso_coefs.T                                   # (n_alphas, n_features), like ridge_coefs

# Plot regularization paths
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for i in range(min(10, ridge_coefs.shape[1])):  # Plot first 10 features
    axes[0,0].plot(alphas_ridge, ridge_coefs[:, i], alpha=0.7)
axes[0,0].set_xscale('log')
axes[0,0].set_xlabel('Alpha')
axes[0,0].set_ylabel('Coefficient Value')
axes[0,0].set_title('Ridge Regularization Path')
axes[0,0].grid(True, alpha=0.3)

for i in range(min(10, lasso_coefs.shape[1])):  # Plot first 10 features
    axes[0,1].plot(alphas_lasso, lasso_coefs[:, i], alpha=0.7)
axes[0,1].set_xscale('log')
axes[0,1].set_xlabel('Alpha')
axes[0,1].set_ylabel('Coefficient Value')
axes[0,1].set_title('Lasso Regularization Path')
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(alphas_ridge, ridge_cv_scores, 'b-', linewidth=2)
axes[1,0].set_xscale('log')
axes[1,0].set_xlabel('Alpha')
axes[1,0].set_ylabel('Cross-validated R² (training data)')
axes[1,0].set_title('Ridge: CV Performance vs Regularization')
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(alphas_lasso, lasso_cv_scores, 'r-', linewidth=2)
axes[1,1].set_xscale('log')
axes[1,1].set_xlabel('Alpha')
axes[1,1].set_ylabel('Cross-validated R² (training data)')
axes[1,1].set_title('Lasso: CV Performance vs Regularization')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Alphas chosen by cross-validation (the test set was not consulted)
best_ridge_idx = int(np.argmax(ridge_cv_scores))
best_lasso_idx = int(np.argmax(lasso_cv_scores))
print(f"Ridge alpha chosen by CV: {alphas_ridge[best_ridge_idx]:.6f} (CV R² = {ridge_cv_scores[best_ridge_idx]:.6f})")
print(f"Lasso alpha chosen by CV: {alphas_lasso[best_lasso_idx]:.6f} (CV R² = {lasso_cv_scores[best_lasso_idx]:.6f})")


### Regularized Regression Insights (from this run):

**What happened here.** In this run linear regression and Ridge give the same test R² (about 0.022, the engineered model of Exercise 3) and the same test RMSE (0.902): cross-validation picked the smallest Ridge alpha in the grid (0.001), i.e. as little shrinkage as possible. Lasso at its grid-search alpha of 0.001 sets 9 of the 32 standardised coefficients to zero and pays for the sparsity with a lower test R² (about 0.014); on the finer path, cross-validation again prefers the smallest alpha tried (0.0001), where Lasso is essentially ordinary least squares. With 33,000 training rows and 32 regressors there is very little variance to trade against bias, so regularisation has almost nothing to do; the regularisation-path plots show the CV score rising as alpha shrinks and levelling off at the smallest alphas, and dropping to zero once alpha is large enough to shrink every coefficient away.

**Key Differences Between Methods:**

1. **Linear Regression:** no regularisation; can overfit when there are many features relative to observations; all features retained.
2. **Ridge Regression (L2):** shrinks coefficients toward zero but does not eliminate them; handles multicollinearity well (it would stabilise the three macro coefficients with VIFs of 30 to 65); good when all features are somewhat relevant.
3. **Lasso Regression (L1):** sets some coefficients exactly to zero, giving a sparse model; good when only a subset of features matters; can be unstable with highly correlated features (it picks one of a collinear group more or less arbitrarily).

**When to Use Each Method:**

- **Linear Regression:** small number of features relative to observations, interpretability crucial (this dataset)
- **Ridge:** many features, multicollinearity, want to keep all features
- **Lasso:** many features, want automatic feature selection, sparse solutions

**A rule that always applies:** the regularisation strength is chosen on training data (cross-validation), never by looking at the test score. The test set is used exactly once, at the end.


## Exercise 8: Business Impact Analysis

**Task:** Quantify the business value of your duration prediction model.

### Solution Approach:
1. Create engagement categories based on call duration
2. Calculate classification accuracy for each category
3. Estimate business value of correct predictions
4. Develop actionable insights

In [ ]:
# Exercise 8 Solution

print("=== BUSINESS IMPACT ANALYSIS ===")

# Define engagement thresholds on the scale of the target, log1p(duration) with duration in SECONDS
low_threshold = np.log1p(120)    # 2 minutes = 120 seconds -> about 4.80
high_threshold = np.log1p(300)   # 5 minutes = 300 seconds -> about 5.71

print(f"Engagement thresholds:")
print(f"Low engagement: < {low_threshold:.3f} (< 2 minutes)")
print(f"Medium engagement: {low_threshold:.3f} - {high_threshold:.3f} (2-5 minutes)")
print(f"High engagement: > {high_threshold:.3f} (> 5 minutes)")

def categorize_engagement(log_duration):
    # Categorize engagement based on log1p(duration)
    return np.where(log_duration < low_threshold, 'Low',
                   np.where(log_duration < high_threshold, 'Medium', 'High'))

# Categorize actual and predicted durations
y_test_categories = categorize_engagement(y_test_banking.ravel())
y_pred_categories = categorize_engagement(y_test_predicted_banking.ravel())

# Distribution of engagement levels: actual vs predicted
print(f"\nRange of the predictions: {y_test_predicted_banking.min():.3f} to {y_test_predicted_banking.max():.3f}")
print(f"(compare: the thresholds are {low_threshold:.3f} and {high_threshold:.3f})")
dist_df = pd.DataFrame({
    'Actual share': pd.Series(y_test_categories).value_counts(normalize=True),
    'Actual count': pd.Series(y_test_categories).value_counts(),
    'Predicted share': pd.Series(y_pred_categories).value_counts(normalize=True),
    'Predicted count': pd.Series(y_pred_categories).value_counts(),
}).reindex(['Low', 'Medium', 'High']).fillna(0)
print("\nActual vs predicted engagement distribution on the test set:")
print(dist_df.to_string(float_format='%.3f'))

# Create confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test_categories, y_pred_categories,
                     labels=['Low', 'Medium', 'High'])

print(f"\nConfusion Matrix:")
cm_df = pd.DataFrame(cm,
                    index=['Actual Low', 'Actual Medium', 'Actual High'],
                    columns=['Pred Low', 'Pred Medium', 'Pred High'])
print(cm_df)

# Calculate accuracy for each engagement level
print(f"\nClassification Report:")
print(classification_report(y_test_categories, y_pred_categories, labels=['Low', 'Medium', 'High'], zero_division=0))


In [ ]:
# Visualize confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues',
            cbar_kws={'label': 'Number of Calls'})
plt.title('Engagement Level Prediction: Confusion Matrix')
plt.ylabel('Actual Engagement')
plt.xlabel('Predicted Engagement')
plt.show()

# Calculate precision, recall, and F1 for each class
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    y_test_categories, y_pred_categories, labels=['Low', 'Medium', 'High'], zero_division=0
)

metrics_df = pd.DataFrame({
    'Engagement Level': ['Low', 'Medium', 'High'],
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
})

print("\nDetailed Metrics by Engagement Level:")
print(metrics_df.to_string(index=False, float_format='%.3f'))

# Visualize metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics_to_plot = ['Precision', 'Recall', 'F1-Score']
colors = ['skyblue', 'lightcoral', 'lightgreen']

for i, metric in enumerate(metrics_to_plot):
    axes[i].bar(metrics_df['Engagement Level'], metrics_df[metric],
               color=colors[i], alpha=0.8)
    axes[i].set_ylabel(metric)
    axes[i].set_title(f'{metric} by Engagement Level')
    axes[i].set_ylim(0, 1)
    axes[i].grid(True, alpha=0.3)

    # Add value labels on bars
    for j, v in enumerate(metrics_df[metric]):
        axes[i].text(j, v + 0.02, f'{v:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()


In [ ]:
# Business value calculation
print("\n=== BUSINESS VALUE ESTIMATION ===")

# Define business values (hypothetical but realistic)
business_values = {
    'cost_per_call': 5.0,  # Cost to make a follow-up call
    'revenue_low': 50.0,   # Expected revenue from low engagement customer
    'revenue_medium': 150.0,  # Expected revenue from medium engagement customer
    'revenue_high': 400.0,    # Expected revenue from high engagement customer
    'conversion_rate_low': 0.05,    # 5% conversion rate for low engagement
    'conversion_rate_medium': 0.15,  # 15% conversion rate for medium engagement
    'conversion_rate_high': 0.35,   # 35% conversion rate for high engagement
}

print("Business assumptions:")
for key, value in business_values.items():
    print(f"{key}: {value}")

# Calculate expected value for each engagement level
expected_values = {
    'Low': business_values['revenue_low'] * business_values['conversion_rate_low'] - business_values['cost_per_call'],
    'Medium': business_values['revenue_medium'] * business_values['conversion_rate_medium'] - business_values['cost_per_call'],
    'High': business_values['revenue_high'] * business_values['conversion_rate_high'] - business_values['cost_per_call']
}

print(f"\nExpected value per follow-up call:")
for level, value in expected_values.items():
    print(f"{level} engagement: ${value:.2f}")

# Strategy 1: Follow up with all customers (baseline)
total_calls_baseline = len(y_test_categories)
baseline_value = sum(expected_values[level] * np.sum(y_test_categories == level) 
                    for level in ['Low', 'Medium', 'High'])

print(f"\nBaseline strategy (follow up with all {total_calls_baseline} customers):")
print(f"Total expected value: ${baseline_value:.2f}")
print(f"Average value per call: ${baseline_value/total_calls_baseline:.2f}")

# Strategy 2: Only follow up with predicted high engagement customers
high_pred_mask = y_pred_categories == 'High'
high_pred_count = np.sum(high_pred_mask)
high_pred_actual = y_test_categories[high_pred_mask]

strategy2_value = sum(expected_values[level] * np.sum(high_pred_actual == level) 
                     for level in ['Low', 'Medium', 'High'])

print(f"\nStrategy 2 (follow up only with predicted high engagement):")
print(f"Calls made: {high_pred_count} ({high_pred_count/total_calls_baseline*100:.1f}% of total)")
print(f"Total expected value: ${strategy2_value:.2f}")
print(f"Average value per call: ${strategy2_value/high_pred_count:.2f}" if high_pred_count > 0 else "No calls made")
print(f"Value vs baseline: ${strategy2_value - baseline_value:.2f}")

# Strategy 3: Follow up with predicted medium and high engagement
med_high_pred_mask = (y_pred_categories == 'Medium') | (y_pred_categories == 'High')
med_high_pred_count = np.sum(med_high_pred_mask)
med_high_pred_actual = y_test_categories[med_high_pred_mask]

strategy3_value = sum(expected_values[level] * np.sum(med_high_pred_actual == level) 
                     for level in ['Low', 'Medium', 'High'])

print(f"\nStrategy 3 (follow up with predicted medium + high engagement):")
print(f"Calls made: {med_high_pred_count} ({med_high_pred_count/total_calls_baseline*100:.1f}% of total)")
print(f"Total expected value: ${strategy3_value:.2f}")
print(f"Average value per call: ${strategy3_value/med_high_pred_count:.2f}" if med_high_pred_count > 0 else "No calls made")
print(f"Value vs baseline: ${strategy3_value - baseline_value:.2f}")

In [ ]:
# ROI analysis and sensitivity testing
print("\n=== ROI ANALYSIS ===")

strategies = {
    'Baseline (All)': {
        'calls': total_calls_baseline,
        'value': baseline_value,
        'cost': total_calls_baseline * business_values['cost_per_call']
    },
    'High Only': {
        'calls': high_pred_count,
        'value': strategy2_value,
        'cost': high_pred_count * business_values['cost_per_call']
    },
    'Medium + High': {
        'calls': med_high_pred_count,
        'value': strategy3_value,
        'cost': med_high_pred_count * business_values['cost_per_call']
    }
}

roi_data = []
for strategy_name, data in strategies.items():
    revenue = data['value'] + data['cost']  # Add back the cost to get gross revenue
    profit = data['value']
    roi = (profit / data['cost']) * 100 if data['cost'] > 0 else 0
    
    roi_data.append({
        'Strategy': strategy_name,
        'Calls': data['calls'],
        'Cost': data['cost'],
        'Revenue': revenue,
        'Profit': profit,
        'ROI (%)': roi,
        'Profit per Call': profit / data['calls'] if data['calls'] > 0 else 0
    })

roi_df = pd.DataFrame(roi_data)
print(roi_df.to_string(index=False, float_format='%.2f'))

# Visualize ROI comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Profit comparison
axes[0,0].bar(roi_df['Strategy'], roi_df['Profit'], alpha=0.8, color='green')
axes[0,0].set_ylabel('Total Profit ($)')
axes[0,0].set_title('Total Profit by Strategy')
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,0].grid(True, alpha=0.3)

# ROI comparison
axes[0,1].bar(roi_df['Strategy'], roi_df['ROI (%)'], alpha=0.8, color='blue')
axes[0,1].set_ylabel('ROI (%)')
axes[0,1].set_title('Return on Investment by Strategy')
axes[0,1].tick_params(axis='x', rotation=45)
axes[0,1].grid(True, alpha=0.3)

# Calls vs Profit
axes[1,0].scatter(roi_df['Calls'], roi_df['Profit'], s=100, alpha=0.8)
for i, strategy in enumerate(roi_df['Strategy']):
    axes[1,0].annotate(strategy, (roi_df['Calls'].iloc[i], roi_df['Profit'].iloc[i]),
                      xytext=(5, 5), textcoords='offset points')
axes[1,0].set_xlabel('Number of Calls')
axes[1,0].set_ylabel('Total Profit ($)')
axes[1,0].set_title('Calls vs Profit')
axes[1,0].grid(True, alpha=0.3)

# Profit per call
axes[1,1].bar(roi_df['Strategy'], roi_df['Profit per Call'], alpha=0.8, color='orange')
axes[1,1].set_ylabel('Profit per Call ($)')
axes[1,1].set_title('Profit per Call by Strategy')
axes[1,1].tick_params(axis='x', rotation=45)
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Sensitivity analysis
print("\n=== SENSITIVITY ANALYSIS ===")

# Test different cost scenarios
cost_scenarios = [2.0, 5.0, 10.0, 15.0]  # Different cost per call
sensitivity_results = []

for cost in cost_scenarios:
    # Recalculate expected values
    exp_val_low = business_values['revenue_low'] * business_values['conversion_rate_low'] - cost
    exp_val_med = business_values['revenue_medium'] * business_values['conversion_rate_medium'] - cost
    exp_val_high = business_values['revenue_high'] * business_values['conversion_rate_high'] - cost
    
    # Baseline strategy value
    baseline_val = (exp_val_low * np.sum(y_test_categories == 'Low') +
                   exp_val_med * np.sum(y_test_categories == 'Medium') +
                   exp_val_high * np.sum(y_test_categories == 'High'))
    
    # High-only strategy value
    high_only_val = (exp_val_low * np.sum(high_pred_actual == 'Low') +
                    exp_val_med * np.sum(high_pred_actual == 'Medium') +
                    exp_val_high * np.sum(high_pred_actual == 'High'))
    
    sensitivity_results.append({
        'Cost per Call': cost,
        'Baseline Profit': baseline_val,
        'High-Only Profit': high_only_val,
        'Improvement': high_only_val - baseline_val,
        'Improvement %': ((high_only_val - baseline_val) / abs(baseline_val)) * 100 if baseline_val != 0 else 0
    })

sensitivity_df = pd.DataFrame(sensitivity_results)
print("Sensitivity to Cost per Call:")
print(sensitivity_df.to_string(index=False, float_format='%.2f'))

# Plot sensitivity
plt.figure(figsize=(12, 6))
plt.plot(sensitivity_df['Cost per Call'], sensitivity_df['Baseline Profit'], 
         'b-o', label='Baseline (All Customers)', linewidth=2)
plt.plot(sensitivity_df['Cost per Call'], sensitivity_df['High-Only Profit'], 
         'r-o', label='High Engagement Only', linewidth=2)
plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
plt.xlabel('Cost per Call ($)')
plt.ylabel('Total Profit ($)')
plt.title('Profit Sensitivity to Cost per Call')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Find break-even point
break_even_cost = business_values['revenue_high'] * business_values['conversion_rate_high']
print(f"\nBreak-even cost per call for high engagement: ${break_even_cost:.2f}")
print(f"Current cost assumption: ${business_values['cost_per_call']:.2f}")
print(f"Safety margin: {((break_even_cost - business_values['cost_per_call']) / business_values['cost_per_call']) * 100:.1f}%")

### Business Impact Analysis Insights (from this run):

**The model cannot rank calls by engagement.** With the correct thresholds (log1p(120) = 4.80 and log1p(300) = 5.71), the actual test calls split into about 31% Low, 41% Medium and 28% High. The *predictions*, however, span only 4.80 to 5.92: about 99.5% of test calls are predicted Medium, about 0.5% (39 calls out of 8,238) are predicted High, and none are predicted Low. The model predicts something close to the average for everyone, which is exactly what a test R² of 0.013 means in practice. Recall for the High class is about 1% and for the Low class 0%.

**What the strategy comparison shows.** "Follow up with everyone" and "follow up with predicted Medium or High" are the same strategy here (the second calls 99.5% of customers). "Follow up only with predicted High" makes 39 calls; 21 of them are genuine High-engagement calls, so the precision of that tiny group is about 54% against a base rate of 28%. The profit per call of that strategy is therefore higher, but it forgoes 99.5% of the customers, and the total profit collapses. No cost-per-call assumption in the sensitivity analysis changes that picture, because the model simply does not separate the groups.

**The honest business conclusion.** Pre-call features do not predict engagement well enough to prioritise follow-ups. What the model can support is capacity planning on averages (for example "landline calls are about 20% shorter" from Exercise 1). Prioritising follow-ups would need information from the call itself (which the leakage rule keeps out of a *pre-call* model, but which is legitimately available for a *post-call* decision) or a different target, such as the subscription outcome `y` with a classification model (the next notebook).

**Implementation lesson.** Always check the distribution of the *predictions* against the thresholds before building a business case on them. A threshold-based strategy on a model whose predictions never cross the thresholds is a strategy that never triggers.


## Exercise 9: Advanced Diagnostics

**Task:** Perform advanced model diagnostics to identify potential issues.

### Solution Approach:
1. Calculate Cook's distance for influential observations
2. Check multicollinearity using Variance Inflation Factor (VIF)
3. Test for autocorrelation in residuals
4. Provide improvement recommendations

In [ ]:
# Exercise 9 Solution

print("=== ADVANCED MODEL DIAGNOSTICS ===")

# First, let's use statsmodels for more detailed diagnostics
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import OLSInfluence
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.stattools import durbin_watson

# Prepare data for statsmodels (add constant for intercept)
X_train_sm = sm.add_constant(X_train_banking)
X_test_sm = sm.add_constant(X_test_banking)

# Fit OLS model using statsmodels
model_sm = sm.OLS(y_train_banking.ravel(), X_train_sm).fit()

print("Model Summary:")
print(model_sm.summary())

# Get predictions and residuals
y_train_pred_sm = model_sm.predict(X_train_sm)
residuals_sm = model_sm.resid

In [ ]:
# 1. Cook's Distance Analysis
print("\n=== COOK'S DISTANCE ANALYSIS ===")

# Influence measures from the leverages (the diagonal of the hat matrix), which OLSInfluence computes in O(n p^2).
# Never build the full n x n hat matrix: with 33,000 rows it would need about 8 GB of memory.
# (OLSInfluence.cooks_distance itself is very slow for n this large, so Cook's distance and the externally
#  studentized residuals are computed from their closed forms below; both are O(n).)
n = len(X_train_sm)
p = X_train_sm.shape[1]
influence = OLSInfluence(model_sm)
leverage = influence.hat_matrix_diag                     # h_i
resid = model_sm.resid                                   # e_i
s2 = model_sm.mse_resid                                  # s^2 = SSE / (n - p)
# Cook's distance: D_i = e_i^2 / (p s^2) * h_i / (1 - h_i)^2
cooks_d = resid**2 / (p * s2) * leverage / (1 - leverage)**2
# Externally studentized residual: t_i = e_i / (s_(i) sqrt(1 - h_i)), with s_(i)^2 the error variance when observation i is left out
s2_not_i = ((n - p) * s2 - resid**2 / (1 - leverage)) / (n - p - 1)
studentized_residuals = resid / np.sqrt(s2_not_i * (1 - leverage))

# Identify influential observations
cooks_threshold = 4 / n  # Common threshold
leverage_threshold = 2 * p / n  # Common threshold

influential_cooks = np.where(cooks_d > cooks_threshold)[0]
high_leverage = np.where(leverage > leverage_threshold)[0]
outliers = np.where(np.abs(studentized_residuals) > 3)[0]  # |t| > 3

print(f"Cook's distance threshold: {cooks_threshold:.6f}")
print(f"Leverage threshold: {leverage_threshold:.6f}")
print(f"Influential observations (Cook's D): {len(influential_cooks)} ({len(influential_cooks)/n*100:.2f}%)")
print(f"High leverage observations: {len(high_leverage)} ({len(high_leverage)/n*100:.2f}%)")
print(f"Outliers (|studentized residual| > 3): {len(outliers)} ({len(outliers)/n*100:.2f}%)")
print(f"Largest Cook's distance: {cooks_d.max():.4f} (a value above 1 would be a serious concern)")

# Plot diagnostic plots (scatter/plot rather than stem: 33,000 stems would take minutes to draw)
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0,0].plot(cooks_d, linestyle='none', marker='.', alpha=0.4)
axes[0,0].axhline(y=cooks_threshold, color='red', linestyle='--', label=f'Threshold ({cooks_threshold:.4f})')
axes[0,0].set_xlabel('Observation Index')
axes[0,0].set_ylabel("Cook's Distance")
axes[0,0].set_title("Cook's Distance")
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

axes[0,1].scatter(range(len(leverage)), leverage, alpha=0.4, s=5)
axes[0,1].axhline(y=leverage_threshold, color='red', linestyle='--', label=f'Threshold ({leverage_threshold:.4f})')
axes[0,1].set_xlabel('Observation Index')
axes[0,1].set_ylabel('Leverage')
axes[0,1].set_title('Leverage Values')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

axes[1,0].scatter(leverage, studentized_residuals, alpha=0.4, s=5)
axes[1,0].axhline(y=3, color='red', linestyle='--', alpha=0.7)
axes[1,0].axhline(y=-3, color='red', linestyle='--', alpha=0.7)
axes[1,0].axvline(x=leverage_threshold, color='red', linestyle='--', alpha=0.7)
axes[1,0].set_xlabel('Leverage')
axes[1,0].set_ylabel('Studentized Residuals')
axes[1,0].set_title('Residuals vs Leverage')
axes[1,0].grid(True, alpha=0.3)

axes[1,1].scatter(leverage, cooks_d, alpha=0.4, s=5)
axes[1,1].axhline(y=cooks_threshold, color='red', linestyle='--', alpha=0.7)
axes[1,1].axvline(x=leverage_threshold, color='red', linestyle='--', alpha=0.7)
axes[1,1].set_xlabel('Leverage')
axes[1,1].set_ylabel("Cook's Distance")
axes[1,1].set_title("Cook's Distance vs Leverage")
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show most influential observations
if len(influential_cooks) > 0:
    print(f"\nTop 10 most influential observations (Cook's Distance):")
    top_influential = np.argsort(cooks_d)[-10:][::-1]
    for i, idx in enumerate(top_influential):
        print(f"{i+1}. Index {idx}: Cook's D = {cooks_d[idx]:.6f}, Leverage = {leverage[idx]:.6f}, Studentized Residual = {studentized_residuals[idx]:.3f}")


In [ ]:
# 2. Multicollinearity Analysis (VIF)
print("\n=== MULTICOLLINEARITY ANALYSIS (VIF) ===")

from statsmodels.stats.outliers_influence import variance_inflation_factor

# X_train_sm is a numpy array (sm.add_constant of a numpy array), so slice it with [:, 1:] to drop the constant column
X_for_vif = X_train_sm[:, 1:]
vif_data = []

for i in range(X_for_vif.shape[1]):
    vif = variance_inflation_factor(X_for_vif, i)
    vif_data.append({
        'Feature': X_df_banking.columns[i],
        'VIF': vif
    })

vif_df = pd.DataFrame(vif_data)
vif_df = vif_df.sort_values('VIF', ascending=False)

print("Variance Inflation Factors:")
print("VIF > 10: High multicollinearity")
print("VIF > 5: Moderate multicollinearity")
print("VIF < 5: Low multicollinearity")
print()
print(vif_df.to_string(index=False, float_format='%.3f'))

# Identify problematic features
high_vif = vif_df[vif_df['VIF'] > 10]
moderate_vif = vif_df[(vif_df['VIF'] > 5) & (vif_df['VIF'] <= 10)]

print(f"\nFeatures with high multicollinearity (VIF > 10): {len(high_vif)}")
if len(high_vif) > 0:
    print(high_vif[['Feature', 'VIF']].to_string(index=False))

print(f"\nFeatures with moderate multicollinearity (5 < VIF <= 10): {len(moderate_vif)}")
if len(moderate_vif) > 0:
    print(moderate_vif[['Feature', 'VIF']].to_string(index=False))

# Visualize VIF
plt.figure(figsize=(12, 8))
colors = ['red' if vif > 10 else 'orange' if vif > 5 else 'green' for vif in vif_df['VIF']]
plt.barh(range(len(vif_df)), vif_df['VIF'], color=colors, alpha=0.7)
plt.yticks(range(len(vif_df)), vif_df['Feature'])
plt.xlabel('Variance Inflation Factor (VIF)')
plt.title('Multicollinearity Analysis: VIF by Feature')
plt.axvline(x=5, color='orange', linestyle='--', alpha=0.7, label='Moderate threshold (5)')
plt.axvline(x=10, color='red', linestyle='--', alpha=0.7, label='High threshold (10)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# 3. Heteroscedasticity Tests (banking model)
print("\n=== HETEROSCEDASTICITY TESTS ===")

# Breusch-Pagan test
bp_stat, bp_pvalue, bp_fstat, bp_fpvalue = het_breuschpagan(residuals_sm, X_train_sm)
print(f"Breusch-Pagan Test:")
print(f"  LM statistic: {bp_stat:.6f}")
print(f"  p-value: {bp_pvalue:.6f}")
print(f"  Interpretation: {'Heteroscedasticity detected' if bp_pvalue < 0.05 else 'No evidence against constant variance'}")

# White test
white_stat, white_pvalue, white_fstat, white_fpvalue = het_white(residuals_sm, X_train_sm)
print(f"\nWhite Test:")
print(f"  LM statistic: {white_stat:.6f}")
print(f"  p-value: {white_pvalue:.6f}")
print(f"  Interpretation: {'Heteroscedasticity detected' if white_pvalue < 0.05 else 'No evidence against constant variance'}")

# 4. Autocorrelation Test (Durbin-Watson)
print("\n=== AUTOCORRELATION TEST (Durbin-Watson) ===")
print("DW = 2: no first-order autocorrelation; DW < 2: positive; DW > 2: negative autocorrelation.")

# 4a. On the banking model the statistic is MEANINGLESS: the rows are individual calls in a random (shuffled) order,
# so 'the previous observation' has no meaning. Shuffling the residuals gives a different but equally meaningless value.
dw_banking = durbin_watson(residuals_sm)
dw_banking_shuffled = durbin_watson(np.random.permutation(residuals_sm))
print(f"\nBanking model (cross-section, arbitrary row order): DW = {dw_banking:.4f}; after shuffling the rows: DW = {dw_banking_shuffled:.4f}")
print("  -> both are close to 2 by construction. Nothing can be concluded from them; the statistic needs a time order.")

# 4b. On the Bitcoin model (Exercise 6) the rows are days, so the statistic has content
btc_model = lag_results[best_lag]
resid_btc_train = btc_model['y_train'] - btc_model['model'].predict(btc_model['X_train'])
dw_btc = durbin_watson(resid_btc_train)
print(f"\nBitcoin model (lag {best_lag}, training residuals in time order): DW = {dw_btc:.4f}")
if abs(dw_btc - 2) < 0.2:
    print("  -> no first-order autocorrelation left in the residuals: the lags have absorbed what little dependence there was.")
elif dw_btc < 2:
    print("  -> positive autocorrelation: the residuals still contain predictable structure.")
else:
    print("  -> negative autocorrelation in the residuals.")

# 5. Normality Tests (banking model; matters for inference in small samples, not for predictions)
print("\n=== ADDITIONAL NORMALITY TESTS ===")

from scipy.stats import jarque_bera, shapiro, normaltest

jb_stat, jb_pvalue = jarque_bera(residuals_sm)
print(f"Jarque-Bera Test:")
print(f"  Statistic: {jb_stat:.6f}")
print(f"  p-value: {jb_pvalue:.6f}")
print(f"  Interpretation: {'Residuals deviate from normality' if jb_pvalue < 0.05 else 'Residuals appear normal'}")

dag_stat, dag_pvalue = normaltest(residuals_sm)
print(f"\nD'Agostino's Normality Test:")
print(f"  Statistic: {dag_stat:.6f}")
print(f"  p-value: {dag_pvalue:.6f}")
print(f"  Interpretation: {'Residuals deviate from normality' if dag_pvalue < 0.05 else 'Residuals appear normal'}")


In [ ]:
# 6. What the diagnostics imply
print("\n=== WHAT THE DIAGNOSTICS IMPLY ===")
print("Assumption checks are about the reliability of the INFERENCE (standard errors, confidence intervals, p-values).")
print("None of them says whether the model is 'safe to use for predictions'; that question is answered by the test-set error.\n")

findings = []

if len(influential_cooks) > len(X_train_sm) * 0.05:  # More than 5% influential
    findings.append(
        f"INFLUENTIAL OBSERVATIONS: {len(influential_cooks)} observations exceed the 4/n Cook's distance threshold "
        f"({len(influential_cooks)/len(X_train_sm)*100:.1f}%), but the largest Cook's D is {cooks_d.max():.3f}, far below 1. "
        "The threshold 4/n is very strict in a sample this large; no single call moves the coefficients."
    )
else:
    findings.append(
        f"INFLUENTIAL OBSERVATIONS: {len(influential_cooks)} observations exceed the 4/n threshold, largest Cook's D = {cooks_d.max():.3f}. "
        "No single call has a material influence on the coefficients."
    )

if len(high_vif) > 0:
    findings.append(
        f"MULTICOLLINEARITY: {len(high_vif)} regressors have VIF > 10 ({', '.join(high_vif['Feature'])}). "
        "Their individual coefficients and standard errors are unreliable; read the macro block jointly, or drop all but one indicator, "
        "or use Ridge. Predictions are unaffected."
    )

if bp_pvalue < 0.05 or white_pvalue < 0.05:
    findings.append(
        "HETEROSCEDASTICITY: the Breusch-Pagan and White tests reject constant variance. With 33,000 observations these tests detect "
        "even small departures. Consequence: use heteroscedasticity-robust standard errors (cov_type='HC1' in statsmodels) before "
        "quoting p-values. The coefficients themselves are still unbiased."
    )

if jb_pvalue < 0.05:
    findings.append(
        "NON-NORMALITY: the residuals have a heavier left tail (very short calls). With this sample size the central limit theorem makes "
        "the confidence intervals valid anyway; the target is already log-transformed, which is the usual remedy."
    )

findings.append(
    f"AUTOCORRELATION: not applicable to the banking cross-section. On the Bitcoin model DW = {dw_btc:.3f}: "
    + ("no residual autocorrelation, so a lagged-return model has nothing more to extract." if abs(dw_btc - 2) < 0.2
       else "residual autocorrelation remains; more lags or a time-series model would be worth trying.")
)

for i, f in enumerate(findings, 1):
    print(f"{i}. {f}\n")

# Summary statistics
print("=== DIAGNOSTIC SUMMARY ===")
summary_stats = {
    'Total Observations': len(X_train_sm),
    'Features': X_train_sm.shape[1] - 1,  # Exclude constant
    'Influential Obs (%)': len(influential_cooks) / len(X_train_sm) * 100,
    'Largest Cook\'s D': cooks_d.max(),
    'High VIF Features': len(high_vif),
    'Heteroscedasticity p-value': min(bp_pvalue, white_pvalue),
    'Durbin-Watson (Bitcoin model)': dw_btc,
    'Normality p-value': jb_pvalue,
    'Model R² (banking, train)': model_sm.rsquared,
    'Adjusted R² (banking, train)': model_sm.rsquared_adj
}

for key, value in summary_stats.items():
    if 'p-value' in key:
        print(f"{key}: {value:.6f}")
    elif '%' in key or 'R²' in key or 'Durbin' in key or 'Cook' in key:
        print(f"{key}: {value:.3f}")
    else:
        print(f"{key}: {value}")


### Advanced Diagnostics Insights:

**Key Diagnostic Tools:**

1. **Cook's Distance:**
   - Measures how much the fitted coefficients change when one observation is removed
   - The common 4/n threshold flags many observations in a large sample; a value above 1 is the level at which one observation really matters
   - Computed from the leverages (the diagonal of the hat matrix); never build the full hat matrix for large n

2. **Variance Inflation Factor (VIF):**
   - Measures multicollinearity between features
   - VIF > 10: high multicollinearity (individual coefficients unreliable)
   - VIF > 5: moderate multicollinearity (worth noting)

3. **Heteroscedasticity Tests:**
   - Breusch-Pagan: tests for variance that depends linearly on the regressors
   - White test: tests for general heteroscedasticity
   - A violation affects standard errors and confidence intervals, not the coefficients; the fix is robust standard errors

4. **Durbin-Watson Test:**
   - Tests for first-order autocorrelation in residuals
   - Only meaningful when the rows have a time order (the Bitcoin model); on a shuffled cross-section it is close to 2 by construction and says nothing
   - Values near 2 indicate no autocorrelation

**What the checks are for:** they tell you how far to trust the standard errors, confidence intervals and p-values of the model. They do not certify a model as "safe for prediction"; the test-set RMSE and R² do that, and for the banking model those say that individual predictions are weak regardless of how well the assumptions hold.

**Common Issues and Solutions:**

1. **Influential Observations:** investigate for data errors; consider robust regression if a few points dominate
2. **Multicollinearity:** read collinear blocks jointly, drop redundant regressors, or use Ridge
3. **Heteroscedasticity:** robust standard errors, weighted least squares, or a transformed target
4. **Non-normality:** a transformed target (already done here with log1p); in large samples the confidence intervals are fine anyway

**Best Practices:**
- Always check assumptions before interpreting coefficients, standard errors and p-values
- Match the check to the data: autocorrelation checks need a time order
- Document any assumption violations and what you did about them


## Reflection Questions - Detailed Answers

### 1. When might linear regression not be appropriate?

**Non-linear relationships:**
- When the relationship between features and target is curved, exponential, or has other non-linear patterns
- Example: Population growth, compound interest, or diminishing returns

**Categorical outcomes:**
- Linear regression predicts continuous values, not categories
- Use logistic regression for binary outcomes, multinomial regression for multiple categories

**Assumption violations:**
- Severe heteroscedasticity (non-constant variance)
- Strong autocorrelation in residuals
- Non-normal residuals with small sample sizes
- Extreme multicollinearity

**Alternative approaches:**
- Polynomial regression for curved relationships
- Tree-based models for complex interactions
- Neural networks for highly non-linear patterns

### 2. How do you balance model complexity with interpretability?

**Start simple:**
- Begin with basic linear model
- Add complexity only when justified by performance gains
- Use statistical tests to validate additional features

**Use regularization:**
- Ridge regression maintains all features but shrinks coefficients
- Lasso automatically selects important features
- Elastic Net combines both approaches

**Business context matters:**
- Regulatory environments may require interpretable models
- High-stakes decisions need explainable predictions
- Operational teams need to understand model logic

**Practical strategies:**
- Create separate models for different purposes (simple for explanation, complex for prediction)
- Use feature importance rankings to focus on key drivers
- Provide model summaries at different technical levels

### 3. What are the key assumptions of linear regression and why do they matter?

**Linearity:**
- Relationship between features and target is linear
- Violation: Biased predictions, poor fit
- Business impact: Wrong understanding of factor relationships

**Independence:**
- Observations are independent of each other
- Violation: Underestimated standard errors, overconfident predictions
- Business impact: False confidence in model reliability

**Homoscedasticity:**
- Constant variance of residuals
- Violation: Unreliable confidence intervals
- Business impact: Incorrect uncertainty estimates for decisions

**Normality:**
- Residuals follow normal distribution
- Violation: Invalid hypothesis tests, poor confidence intervals
- Business impact: Unreliable statistical inference

**No multicollinearity:**
- Features are not highly correlated
- Violation: Unstable coefficients, difficult interpretation
- Business impact: Wrong conclusions about factor importance

### 4. How would you explain R² to a non-technical business stakeholder?

**Simple explanation:**
"R² tells us what percentage of the variation in our target variable is explained by our model. It's like asking: 'How much of the ups and downs in our data can we predict using our features?'"

**Practical interpretation:**
- R² = 0.80 means "Our model explains 80% of why values vary"
- R² = 0.30 means "Our model captures 30% of the pattern, 70% is due to other factors"

**Business context:**
- Higher R² = More predictable outcomes
- Lower R² = More uncertainty, need additional factors
- Perfect R² (1.0) is rare in real business data

**Avoid common misconceptions:**
- R² doesn't prove causation
- Higher R² doesn't always mean better business decisions
- R² can be misleading with small samples or many features

### 5. In what business scenarios would you prefer RMSE over R² as an evaluation metric?

**When absolute errors matter:**
- Financial forecasting: $1000 error has real cost regardless of scale
- Inventory management: Overstocking/understocking has direct costs
- Resource planning: Wrong headcount predictions affect operations

**When comparing models with different scales:**
- RMSE has same units as target variable
- Easier to interpret business impact
- Can set acceptable error thresholds

**When stakeholders need concrete numbers:**
- "Average error is $500" vs "Model explains 85% of variance"
- RMSE directly relates to business costs
- Easier to set performance targets

**Examples:**
- Sales forecasting: RMSE shows average dollar error
- Demand planning: RMSE indicates typical unit shortage/surplus
- Budget planning: RMSE reveals expected deviation from targets

### 6. How might you improve the Bitcoin forecasting model, and what is the most likely outcome of trying?

**Set expectations first:** Exercise 6 showed that lagged returns carry no usable information about the next day's return (RMSE ratio above 1 against the zero forecast, directional accuracy within noise of the share of up-days). That is what weak-form market efficiency predicts, and the most likely outcome of any of the ideas below is a model that still does not beat the zero forecast out of sample after costs. Ideas that are worth trying are those that add information that is not already in past prices.

**Additional features:**
- Market sentiment indicators (fear/greed index)
- Trading volume and volatility measures
- Macroeconomic indicators (interest rates, inflation)
- Social media sentiment and news analysis

**Advanced techniques:**
- GARCH models for volatility clustering (volatility, unlike the return itself, is forecastable; this is the realistic target)
- ARIMA models for time series patterns
- Ensemble methods combining multiple models
- Deep learning for complex pattern recognition

**Risk management:**
- Implement position sizing based on prediction confidence
- Add stop-loss and take-profit mechanisms
- Consider transaction costs and market impact
- Regular model retraining and performance monitoring

**Practical considerations:**
- Real-time data feeds for timely predictions
- Backtesting with realistic trading constraints
- Stress testing under different market conditions
- Integration with existing trading infrastructure

## Final Takeaways

**Linear regression strengths:**
- Simple, interpretable, and fast
- Good baseline for more complex models
- Well-understood statistical properties
- Effective when assumptions are met

**Key success factors:**
- Always check and validate assumptions
- Focus on business value, not just statistical metrics
- Use appropriate evaluation methods (cross-validation)
- Consider model limitations in decision-making

**Best practices:**
- Start simple, add complexity gradually
- Document assumptions and limitations
- Regular model monitoring and updates
- Clear communication with stakeholders

**Remember:** The best model is not always the most complex one, but the one that provides reliable, actionable insights for business decisions while being appropriately validated and understood by its users.